# Trabajo de investigación

## Entrada de Chancay y desconcentración del tráfico de contenedores en la costa central peruana

**Universidad Nacional del Centro del Perú — Facultad de Economía**  
**Teoría de la Regulación Económica**  
**Docente — Dr. Ciro Iván Machacuay Meza**

**Integrantes**  
Lizzie Gimena Bartolo Artica  
Patricia Goetendia Limache  
Bridgit Manani Carrión  
Joseph Manuel Orellana Allpoc

Huancayo, octubre de 2026

## SELECCIONA LA CARPETA DONDE QUIERES QUE SE ALMACENE LA INVESTIGACIÓN

En la siguiente celda se establece la ubicación de trabajo. Allí se guardarán los documentos oficiales descargados, sus tablas procesadas, la base mensual, las estimaciones, las figuras y las comprobaciones de procedencia.

In [ ]:
from pathlib import Path
import os

def seleccionar_carpeta():
    # La ruta declarada como variable de entorno permite ejecuciones no interactivas.
    variable = os.environ.get('HHI_DIR_SALIDA', '').strip()
    if variable:
        elegida = Path(variable).expanduser()
    else:
        try:
            import tkinter as tk
            from tkinter import filedialog
            ventana = tk.Tk()
            ventana.withdraw()
            ventana.attributes('-topmost', True)
            carpeta = filedialog.askdirectory(title='SELECCIONA LA CARPETA DONDE QUIERES QUE SE ALMACENE LA INVESTIGACION', mustexist=True, parent=ventana)
            ventana.destroy()
            if not carpeta:
                raise RuntimeError('No se seleccionó una carpeta. Ejecuta nuevamente esta celda.')
            elegida = Path(carpeta)
        except Exception as error:
            print('No fue posible abrir la ventana:', error)
            ruta = input('Escribe una ruta completa para la investigación: ').strip().strip('"')
            if not ruta:
                raise ValueError('Necesitas indicar la carpeta donde se guardarán los datos.')
            elegida = Path(ruta).expanduser()
    elegida.mkdir(parents=True, exist_ok=True)
    if not elegida.is_dir():
        raise NotADirectoryError(elegida)
    return elegida.resolve()
CARPETA_RAIZ = seleccionar_carpeta()
print('Carpeta seleccionada:', CARPETA_RAIZ)

## 1. Delimitación de la investigación y procedimiento

La base comprende enero de 2015 a julio de 2026, con el mes como unidad de observación. Se utiliza el tráfico en TEU de APM Terminals Callao, DP World Callao y Chancay para estudiar el índice de Herfindahl–Hirschman (HHI); Paita se incorpora en un ejercicio de sensibilidad geográfica. Las series del Banco Central de Reserva del Perú permiten controlar variaciones del comercio exterior.

Las fuentes se obtienen **desde las páginas oficiales de la Autoridad Portuaria Nacional (APN) y desde la API pública del BCRP**. La extracción web comienza con la identificación de documentos en las páginas institucionales; después se descargan sus archivos originales. Se comprueba el tipo de archivo, se calcula SHA-256 y se registra la URL y la fecha de consulta. Ninguna base de tráfico ni respuesta de API está incorporada dentro del cuaderno.

La muestra principal conserva sin HHI conjunto noviembre de 2024–mayo de 2025, periodo de transición anterior al tramo comercial definido desde junio de 2025. Se reconstruyen 132 observaciones válidas dentro de un calendario de 139 meses. Las transformaciones y las estimaciones se calculan exclusivamente con los datos recuperados al ejecutar el cuaderno.

In [ ]:
CONFIG = {
    'inicio': '2015-01',
    'fin': '2026-07',
    'transicion': '2024-11',
    'comercial': '2025-06',
    'inicio_ventana': '2021-01',
    'comparacion_meses': [6, 7, 8, 9, 10],
    'meses_covid': ('2020-03', '2021-12'),
    'meses_confinamiento': ('2020-03', '2020-06'),
    'retardo_hac': 12,
    'retardo_ljungbox': 12,
    'max_iteraciones': 400,
    'pausa_segundos': 1.0,
    'intentos': 3,
    'usar_selenium': True,
    'comprobar_sha_archivado': True,
    'codigos_bcrp': {
        'PN38714BM': 'exportaciones_fob_mill_usd',
        'PN38718BM': 'importaciones_fob_mill_usd',
        'PN38727BM': 'var_interanual_volumen_exportado_pct',
        'PN38728BM': 'var_interanual_volumen_importado_pct',
    },
}

## 2. Bibliotecas y organización de los archivos

Se comprueba la importación de los paquetes utilizados para la extracción, el procesamiento estadístico y la estimación econométrica. Si una biblioteca está incompleta, se recupera únicamente la que presenta el problema, en una carpeta del proyecto. La comprobación se ejecuta en el intérprete del notebook y deja constancia de las versiones utilizadas.

In [ ]:
import os
import sys
import json
import re
import subprocess
import importlib
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Las comprobaciones se ejecutan fuera del kernel para evitar módulos parcialmente importados.
DEPENDENCIAS = CARPETA_RAIZ / '00_dependencias_python'
REGISTRO_DEPENDENCIAS = CARPETA_RAIZ / 'registro_dependencias.txt'

PRUEBA_IMPORTACION = r"""
import importlib, json, traceback, warnings
warnings.filterwarnings('ignore', message=r'Pandas requires version .*numexpr', category=UserWarning)
MODULOS = {
    'numpy': 'numpy',
    'scipy': 'scipy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib.pyplot',
    'statsmodels': 'statsmodels.api',
    'openpyxl': 'openpyxl',
    'pymupdf': 'pymupdf',
    'requests': 'requests',
    'beautifulsoup4': 'bs4',
    'threadpoolctl': 'threadpoolctl',
    'IPython': 'IPython',
}
resultado = {}
for paquete, modulo in MODULOS.items():
    try:
        importlib.import_module(modulo)
        if paquete == 'statsmodels':
            from statsmodels.tsa.statespace.sarimax import SARIMAX
            from statsmodels.stats.diagnostic import acorr_ljungbox
        elif paquete == 'matplotlib':
            import matplotlib.dates
            import matplotlib.backends
        resultado[paquete] = {'ok': True, 'detalle': ''}
    except Exception:
        resultado[paquete] = {'ok': False, 'detalle': traceback.format_exc(limit=9)[-1900:]}
print('RESULTADO_DEPENDENCIAS_HHI=' + json.dumps(resultado, ensure_ascii=False))
"""

REQUISITOS = {
    'numpy': 'numpy>=1.26',
    'scipy': 'scipy>=1.11',
    'pandas': 'pandas>=2.2',
    'matplotlib': 'matplotlib>=3.9,<4',
    'statsmodels': 'statsmodels>=0.14.4,<0.15',
    'openpyxl': 'openpyxl>=3.1',
    'pymupdf': 'pymupdf>=1.23',
    'requests': 'requests>=2.31',
    'beautifulsoup4': 'beautifulsoup4>=4.12',
    'threadpoolctl': 'threadpoolctl>=3.1',
    'IPython': 'IPython>=8',
    'selenium': 'selenium>=4.20,<5',
}


def entorno_de_prueba():
    entorno = os.environ.copy()
    if DEPENDENCIAS.is_dir():
        previo = entorno.get('PYTHONPATH', '')
        entorno['PYTHONPATH'] = str(DEPENDENCIAS) + (os.pathsep + previo if previo else '')
    return entorno


def comprobar_bibliotecas():
    proceso = subprocess.run(
        [sys.executable, '-c', PRUEBA_IMPORTACION],
        capture_output=True, text=True, encoding='utf-8', errors='replace',
        env=entorno_de_prueba(), timeout=150,
    )
    prefijo = 'RESULTADO_DEPENDENCIAS_HHI='
    lineas = [line[len(prefijo):] for line in proceso.stdout.splitlines()
              if line.startswith(prefijo)]
    if not lineas:
        raise RuntimeError('No fue posible completar el diagnóstico de Python.\n'
                           + (proceso.stderr or proceso.stdout)[-2200:])
    return json.loads(lineas[-1])


def activar_dependencias():
    if DEPENDENCIAS.is_dir() and str(DEPENDENCIAS) not in sys.path:
        sys.path.insert(0, str(DEPENDENCIAS))
        importlib.invalidate_caches()


def instalar_paquete(paquete, especificacion=None):
    # Se conserva Anaconda sin modificar; las reparaciones se guardan en el proyecto.
    DEPENDENCIAS.mkdir(parents=True, exist_ok=True)
    requisito = especificacion or REQUISITOS[paquete]
    comando = [sys.executable, '-m', 'pip', 'install',
               '--disable-pip-version-check', '--no-input', '--no-cache-dir',
               '--only-binary=:all:', '--no-deps', '--upgrade', '--retries', '2',
               '--target', str(DEPENDENCIAS), requisito]
    print('Revisando instalación de', paquete + '...')
    try:
        proceso = subprocess.run(comando, capture_output=True, text=True,
                                 encoding='utf-8', errors='replace', timeout=360,
                                 env=entorno_de_prueba())
        salida = (proceso.stdout or '') + '\n' + (proceso.stderr or '')
        codigo = proceso.returncode
    except (OSError, subprocess.TimeoutExpired) as error:
        salida, codigo = repr(error), 1
    with REGISTRO_DEPENDENCIAS.open('a', encoding='utf-8') as archivo:
        archivo.write('\n=== ' + paquete + ' | ' + requisito + ' ===\n' + salida + '\n')
    if codigo != 0:
        print('La instalación de', paquete, 'no se completó.')
        print(salida[-1600:])
    activar_dependencias()
    return codigo == 0


def fallas(estado):
    return [clave for clave, dato in estado.items() if not dato['ok']]


if sys.version_info < (3, 9):
    raise RuntimeError('Se requiere Python 3.9 o posterior.')

activar_dependencias()
MODULOS_GRAFICOS_PREVIOS = any(nombre == 'matplotlib' or nombre.startswith('matplotlib.')
                              for nombre in sys.modules)
matplotlib_reparado = False
estado = comprobar_bibliotecas()

# Matplotlib se atiende primero porque una importación incompleta también puede afectar a Statsmodels.
if not estado['matplotlib']['ok']:
    print('Se detectó una instalación incompleta de Matplotlib.')
    try:
        version_mpl = version('matplotlib')
        version_mpl = version_mpl if re.fullmatch(r'\d+\.\d+\.\d+', version_mpl) else None
    except PackageNotFoundError:
        version_mpl = None
    especificacion = 'matplotlib==' + version_mpl if version_mpl else REQUISITOS['matplotlib']
    if not instalar_paquete('matplotlib', especificacion):
        # Cuando no se encuentra el wheel de la versión anterior, se solicita una compatible.
        instalar_paquete('matplotlib', REQUISITOS['matplotlib'])
    estado = comprobar_bibliotecas()
    if not estado['matplotlib']['ok']:
        raise RuntimeError(
            'Matplotlib sigue sin funcionar. Comprueba la conexión a Internet y revisa '
            + str(REGISTRO_DEPENDENCIAS) + '\n'
            + estado['matplotlib']['detalle'][-1100:]
        )
    matplotlib_reparado = True
    print('Matplotlib se importa correctamente.')

# Solo los paquetes que continúen fallando se revisan de manera individual.
for paquete in [nombre for nombre in fallas(estado) if nombre != 'matplotlib']:
    instalar_paquete(paquete)
    estado = comprobar_bibliotecas()

pendientes = fallas(estado)
if pendientes:
    mensajes = '\n'.join(p + '\n' + estado[p]['detalle'][-700:] for p in pendientes)
    raise RuntimeError('No fue posible recuperar estas bibliotecas.\n' + mensajes
                       + '\nRegistro de instalación: ' + str(REGISTRO_DEPENDENCIAS))

# Las bibliotecas de cálculo quedan disponibles en el kernel activo.
activar_dependencias()
if matplotlib_reparado and MODULOS_GRAFICOS_PREVIOS:
    raise RuntimeError('La reparación se instaló correctamente, pero Matplotlib ya estaba cargado. Reinicia el kernel y '
                       'ejecuta todas las celdas desde la primera. La reparación quedó guardada.')

# NumExpr es opcional; las operaciones estadísticas pueden continuar sin él.
if CONFIG['usar_selenium'] and os.environ.get('HHI_PRUEBA_SIN_NAVEGADOR') != '1':
    try:
        import selenium
    except ImportError:
        if not instalar_paquete('selenium'):
            print('Selenium no está disponible. Continúa el descubrimiento mediante HTTP.')

print('Bibliotecas verificadas. Entorno de Python:', sys.executable)


VERSIONES_USADAS = []
for nombre in ('numpy','pandas','scipy','matplotlib','statsmodels','openpyxl','pymupdf','requests','beautifulsoup4','threadpoolctl','selenium'):
    try:
        VERSIONES_USADAS.append(f'{nombre}=={version(nombre)}')
    except PackageNotFoundError:
        pass
(CARPETA_RAIZ/'versiones_bibliotecas.txt').write_text('\n'.join(VERSIONES_USADAS)+'\n', encoding='utf-8')


In [ ]:
import re, json, hashlib, time, warnings, io
warnings.filterwarnings('ignore', message=r'Pandas requires version .*numexpr', category=UserWarning)
from datetime import datetime, timezone, timedelta
from urllib.parse import urljoin, urlparse, urlsplit, unquote
from urllib.robotparser import RobotFileParser
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError
import requests
import numpy as np
import pandas as pd
pd.set_option('compute.use_numexpr', False)
from bs4 import BeautifulSoup
from openpyxl import load_workbook, Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
import pymupdf as fitz
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from threadpoolctl import threadpool_limits
from IPython.display import display
threadpool_limits(limits=1)
try:
    ZONA_HORARIA_LIMA = ZoneInfo('America/Lima')
except ZoneInfoNotFoundError:
    ZONA_HORARIA_LIMA = timezone(timedelta(hours=-5))

ORIG = CARPETA_RAIZ / '01_fuentes_oficiales'
ORIG_APN = ORIG / 'APN'
ORIG_BCRP = ORIG / 'BCRP'
CATALOGOS = CARPETA_RAIZ / '02_catalogos'
PROCESADOS = CARPETA_RAIZ / '03_bases_procesadas'
RESULTADOS = CARPETA_RAIZ / '04_estimaciones'
FIGURAS = CARPETA_RAIZ / '05_figuras'
WEB = CARPETA_RAIZ / '06_fuentes_para_web'
WEB_EXCEL = WEB / 'excel'
AUDITORIA = CARPETA_RAIZ / '07_verificacion'
for p in [ORIG_APN, ORIG_BCRP, CATALOGOS, PROCESADOS, RESULTADOS, FIGURAS, WEB_EXCEL, AUDITORIA]:
    p.mkdir(parents=True, exist_ok=True)
print('Directorio de investigación:', CARPETA_RAIZ)

## 3. Identificación de publicaciones oficiales

Se trabaja con el libro histórico de TEU de 2010–2021, los 24 resúmenes mensuales en Excel de 2022–2023 y los 31 reportes PDF de contenedores de 2024–julio de 2026. El listado de direcciones constituye un **catálogo de procedencia**, no una copia de los datos. Los documentos se solicitarán de nuevo por HTTP y quedarán guardados en la carpeta elegida. Para identificar los enlaces de las publicaciones se consulta el HTML con BeautifulSoup y, cuando corresponda, con Selenium. Se conservarán las huellas archivadas para comprobar que no haya cambiado la versión publicada.

In [ ]:
from io import StringIO
METADATOS_FUENTES = 'periodo\ttipo\turl\tsha256\n\thistorico_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/3030313/Movimiento%20de%20contenedores%20mensualizado%20a%20nivel%20nacional%20-%202010%20al%202021.xlsx?v=1689360778\td44d27ed8252c9fda0a1a8c1eaf52f58e3c8087a9d295f933b809a94ea1979d7\n2022-01\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505568/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-enero-2022.xlsx?v=1701378186\te5cce27bac54f8390f681045f864bb3774c67cbea264f5379a2ea1cac9f2ca7a\n2022-02\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505569/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-febrero-2022.xlsx?v=1701378295\ta7f40534a50d5727f406bccc3560f23cf5a0b4a1b5023b1654fafa164c759da3\n2022-03\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505570/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-marzo-2022.xlsx?v=1701378402\t2bf0e3d9a196e3ffacbd4aa7c0164c12ec4283733ccfdfa8a5c784d1cfc46832\n2022-04\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505687/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-abril-2022.xlsx?v=1701380525\t161591eac4895b7a2104a329145faa79d317fac7da5d8b58013a97fd2b48d629\n2022-05\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505688/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-mayo-2022.xlsx?v=1701380546\t47ef154ba1361e29eea38a37de8e6ab043ca13ec127fa913b5fef45a04a1178b\n2022-06\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505689/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-junio-2022.xlsx?v=1701380648\t31b54b6024e42367baca3ade9c0f19810d137aab5772b5e9625056e1ecd792a3\n2022-07\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505690/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-julio-2022.xlsx?v=1701380758\t3bfee32b646877924d1cfe785ab33b39f53cda07bc61bb442d9e711b8ad87423\n2022-08\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505691/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-agosto-2022.xlsx?v=1701380856\t7e155d4fbc8c2b124a58b3dc18def79382c7c9668e147589532eeb578d0b85ea\n2022-09\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505692/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-setiembre-2022.xlsx?v=1701380944\tcf2a12096ca3fdb489753c8cd41b2df38d24dbe0dcb493876ac9fb8d4d817702\n2022-10\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505693/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-octubre-2022.xlsx?v=1701381028\t96c9aa167b83fdd318ce339a3abc41a40d8bfa55a1134d4f638b5157b3c4b77a\n2022-11\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505694/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-noviembre-2022.xlsx?v=1701381112\t1d9ad180c0190d8f77f058b40aa29c1efe02dd5b539e0aa105b216dccbce44b0\n2022-12\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5505714/2804703-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-diciembre-2022.xlsx?v=1701381180\te0b9e9303c3dbb045286f9a8db71d065c9f74427f2916a9514ec99ad0531f064\n2023-01\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537905/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-enero-2023.xlsx?v=1702334719\td993b292c1ccb70292fa695e828eb0ac754ec3c9064c7614d21f84edb3035f29\n2023-02\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537906/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-febrero-2023.xlsx?v=1702334776\tf7d4b703989a6c73489b63ac3976b45712f6c6cb9afd054205a170095ba34be9\n2023-03\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537907/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-marzo-2023.xlsx?v=1702334836\t57b05a4b13c033ce372a491bcb2069bf2ef4a400d1dc5d7b145c9c55b8c200d6\n2023-04\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537908/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-abril-2023.xlsx?v=1702334879\t0ff5bfe301ca1c8d2abeebff7b157dab5723e1cbb5e33dac13bb294fe8a717ba\n2023-05\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537909/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-mayo-2023.xlsx?v=1702334915\t19021c89cf4cc8b28611a66a10eb19e7efc96bb9e7ed215b843f6fa592006de5\n2023-06\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537910/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-junio-2023.xlsx?v=1702334943\ta0b59ebb7fac9eee9a2044ce8a224f16bf7c2ffddc84885db1dfab1573b43d18\n2023-07\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537911/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-julio-2023.xlsx?v=1702334961\td072031299c644550bc4ec001be159294abbc61f725fb0a64f9334296205a570\n2023-08\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537912/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-agosto-2023.xlsx?v=1702334972\t2530277cd2ab2f8829e37804940a1e244f80b18341dd2cf27757b1a00296e370\n2023-09\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5537913/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-setiembre-2023.xlsx?v=1703776770\tc837fd6b73c40376e9bec7342f615ba0e8a8d9342f37dae33723d67adc8a59ce\n2023-10\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5610359/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-octubre-2023.xlsx?v=1703776771\t86ebf7b7d18186b62d9032d8ef7e46f5a1ac595d22a7b7ad375bfa31ca2a93fa\n2023-11\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5899143/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-noviembre-2023.xlsx?v=1708548423\t15494bc8056b64909d4287ec5f1cebd1207307e260579d7c60ac6aa846fc8712\n2023-12\tresumen_xlsx\thttps://cdn.www.gob.pe/uploads/document/file/5899145/3986984-resumen-del-movimiento-de-carga-en-los-terminales-portuarios-de-uso-publico-diciembre-2023.xlsx?v=1708548424\tc55c1e123d6d52fff95205bfabe14236d6a284f72b0270bf91f52b79db834dba\n2024-01\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6144485/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-enero-2024.pdf?v=1712250012\t19c15277faaae7ca72a9ef635f4d26465158cad54c3e5752bf8001e7deba7957\n2024-02\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6143447/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-febrero-2024.pdf?v=1712250000\tde85777d799e88d9796ae84525891a7788293d29db91f0bfe92ba3beb8ca6c6e\n2024-03\tcontenedores_pdf\thttps://piep.apn.gob.pe/storage/2024/06/Reporte-de-carga-Marzo-2024-TEUS.pdf\t17557ab16f9ad3ebd266a2369c6569acebd5c43f5468b82317b0366e882425d4\n2024-04\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6465405/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-abril-2024.pdf?v=1718134562\t3d5822127fd9b71de18dfdf427a94bbd271a59d1be9308e0a369fafe61157427\n2024-05\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6669634/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-mayo-2024.pdf?v=1721323083\tb79dfbbba01d6b2decaa8ef1674f4866b583b9e767d42fd884271e0ef5873a10\n2024-06\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6824445/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-junio-2024.pdf?v=1724355940\t67f6f7faf6364bfe68d360efa620f559b4b540b4689e73c44f03041febc66cd5\n2024-07\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/6876476/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-julio-2024.pdf?v=1725313182\t00988e40f810fdbdc40d88c5a463ca2770ed64ee1436f3fba28144ac12427f18\n2024-08\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7297505/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-agosto-2024.pdf?v=1732810481\t8f1dca2732230070400fcf85213370c60ecc16644668595104fb7d499a69f03a\n2024-09\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7297585/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-setiembre-2024.pdf?v=1732810934\tf0d13c07b2a5d53f30b61e3140f3c1e06ddb8fc0f567d2abca3c6f7e38b658b3\n2024-10\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7392879/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-octubre-2024.pdf?v=1734545275\t8acaaab3b71245eee23c81493db9ceb286cedcd1e6fe9f09d2fb3cc3b0ee1d17\n2024-11\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7655431/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-noviembre-2024%282%29.pdf?v=1739976575\t2b92fff284de5b89f3352770849c7a890c4d684717f1247f5fd6c2beddc537f1\n2024-12\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7656430/5425311-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-diciembre-2024.pdf?v=1739981007\tabce12a3a016632eef807d400d1e70ea75db87027b8beedf419256329749c36c\n2025-01\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/7783523/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-enero-2025.pdf?v=1742222410\t7f82d6ac555385950a4d0774277b601b18e8ad97d957b3cb85d9663c0cba789e\n2025-02\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8133807/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-febrero-2025.pdf?v=1748449938\t8a0e4fda44ba6eda1cab0a1d3dab63748535e8667bedcf3ca0130f3f82be278b\n2025-03\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8133810/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-marzo-2025.pdf?v=1748449940\t5bba4b8178b0b9b7c94e899aafc5ccf185dda942eeb3bd2e3088ff2121b1eeb3\n2025-04\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8188129/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-abril-2025.pdf?v=1749491317\t8554478939584de8f9edafe1153fe93a862a3ffd1c788a541a798c5bf91facda\n2025-05\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8324141/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-mayo-2025.pdf?v=1751893897\t1d316ae606bf12a3295b6c2922491965adfa2c290887a705dd5091a2ec37a96c\n2025-06\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8462381/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-junio-2025.pdf?v=1754681188\t272435f0c46d07af050edd91e98b693209442e736723852ac49793e3263989cf\n2025-07\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8592710/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-julio-2025.pdf?v=1756910786\t95e335457e86522b959599bb4ab00d0ea209ea8d8c285a420b71b448742801ae\n2025-08\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8843486/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-agosto-2025.pdf?v=1760712257\t85dfd9489d8cc884bad7554fc87a681f51ad4ae5ea3474bbe6d8436ab0658ec5\n2025-09\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/8957096/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-septiembre-2025.pdf?v=1762466071\ta1a7c73ef976196c40cdb349d4d222e1f2503afd7949a72780f88af32e09172a\n2025-10\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/9089690/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-octubre-2025.pdf?v=1764707447\t9f373007e1e3a545f870a5707bf19d7ea0210f62dc8f81130ecba19d9113db7f\n2025-11\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/9260719/6573509-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-noviembre-2025.pdf?v=1767881340\tb80e127e88cb8d63b8bf085b06bf066941f8d7ddbbcbd6b4a627ff9384b7438d\n2025-12\tcontenedores_pdf\thttps://piep.apn.gob.pe/storage/2026/02/Reporte-de-Contenedores-Diciembre-2025.pdf\t53f15012959f076cba1e3bc2f87300c639c1c63772ee02b4107f6230cfe2f842\n2026-01\tcontenedores_pdf\thttps://piep.apn.gob.pe/storage/2026/03/Reporte-de-Contenedores-Enero-2026.pdf\tc2ef9733f14ff8ff03d1c94662b458147aea2928f6dae047d0da1fdfdd8110a5\n2026-02\tcontenedores_pdf\thttps://piep.apn.gob.pe/storage/2026/03/Reporte-de-Contenedores-Febrero-2026.pdf\tf1571c8af6909962e415a420af06d535832cf37a0620d5b900964458270c5741\n2026-03\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/9974708/7934987-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-marzo-2026.pdf?v=1778857015\td53a22e4c8ffb990b1201ed2f5c75027eb16c5f340af30886d5a2b7404e0fe86\n2026-04\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/10176104/7934987-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-abril-2026.pdf?v=1781798698\ta509f40eba5a548656a99a1aee1d0028df0be506d66e3c0a0e3b12bd6758c639\n2026-05\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/10313500/7934987-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-mayo-2026.pdf?v=1784065598\tc1fe09fee62423e2623b23644902569202df9a39217d7701e1a52d6526bb887b\n2026-06\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/10410407/7934987-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-junio-2026.pdf?v=1785965336\t6439cd17ffcf50f9ad4c45d3954f2693493a5c0c9530a2fd082a11d8f43bb5ff\n2026-07\tcontenedores_pdf\thttps://cdn.www.gob.pe/uploads/document/file/10583235/7934987-movimiento-de-contenedores-en-los-terminales-portuarios-de-uso-publico-a-nivel-nacional-julio-2026.pdf?v=1788815694\tc082aae882e547fd31dffcd55c1a59c44e5ee14632a719d33c6fe76f82d1135e'
FUENTES_APN = pd.read_csv(StringIO(METADATOS_FUENTES), sep='\t', dtype=str).fillna('')
assert len(FUENTES_APN) == 56 and FUENTES_APN.url.str.startswith('https://').all()

PAGINAS_APN = {
    2022: 'https://www.gob.pe/institucion/apn/informes-publicaciones/2804703-estadisticas-apn-2022-trafico-de-carga',
    2023: 'https://www.gob.pe/institucion/apn/informes-publicaciones/3986984-estadisticas-apn-2023-trafico-de-carga',
    2024: 'https://www.gob.pe/institucion/apn/informes-publicaciones/5425311-estadisticas-apn-2024-trafico-de-carga',
    2025: 'https://www.gob.pe/institucion/apn/informes-publicaciones/6573509-estadisticas-apn-2025-trafico-de-carga',
    2026: 'https://www.gob.pe/institucion/apn/informes-publicaciones/7934987-estadisticas-apn-2026-trafico-de-carga',
}
PAGINAS_APN['historico'] = 'https://piep.apn.gob.pe/publicaciones-estadisticas/'
FUENTES_APN.to_csv(CATALOGOS / 'fuentes_apn_previstas.csv',index=False,encoding='utf-8-sig')
print('Fuentes APN identificadas:',len(FUENTES_APN))

## 4. Consulta web, Selenium y descarga de documentos

El navegador automatizado permite identificar enlaces disponibles después de cargar el contenido de una página. Para descargar los PDF y Excel se utiliza HTTP con pausas, reintentos y comprobación del formato recibido. El catálogo institucional previamente identificado permite solicitar directamente una dirección concreta cuando el portal no expone la misma ruta en el HTML. Se registran tanto los enlaces encontrados en la página como los archivos efectivamente descargados. No se emplean técnicas para eludir restricciones de acceso.

In [ ]:
SESION = requests.Session()
SESION.headers.update({'User-Agent': 'Investigacion-Portuaria-UNCP/1.0 (estadisticas publicas)', 'Accept': '*/*'})
LOG = []
ROBOTS = {}

def evento(etapa, url, estado, detalle=''):
    LOG.append({'momento_lima': datetime.now(ZoneInfo('America/Lima')).isoformat(),
                'etapa': etapa, 'url': url, 'estado': str(estado), 'detalle': str(detalle)[:350]})

def robots_permite(url):
    partes = urlsplit(url)
    origen = f'{partes.scheme}://{partes.netloc}'
    if origen not in ROBOTS:
        robots_url = origen + '/robots.txt'
        try:
            r = SESION.get(robots_url, timeout=15)
            evento('robots',robots_url,r.status_code)
            if r.status_code == 200:
                rp = RobotFileParser(); rp.parse(r.text.splitlines()); ROBOTS[origen] = rp
            elif r.status_code == 404:
                ROBOTS[origen] = True
            else:
                ROBOTS[origen] = None
        except requests.RequestException as exc:
            ROBOTS[origen] = None
            evento('robots',robots_url,'no_disponible',type(exc).__name__)
    politica = ROBOTS[origen]
    if politica is None:
        print(f'Advertencia: no fue posible verificar robots.txt en {origen}. Revisar sus términos antes de ejecutar descargas masivas.')
        return True  # Se consultan solo documentos estadísticos públicos, sin eludir autenticación.
    return politica is True or politica.can_fetch(SESION.headers['User-Agent'],url)

def solicitar(url, etapa, destino=None):
    if not robots_permite(url):
        raise PermissionError(f'El portal no permite la consulta automatizada de {url}')
    ultimo = None
    for intento in range(CONFIG['intentos']):
        try:
            time.sleep(CONFIG['pausa_segundos'])
            r = SESION.get(url,timeout=(12,50),allow_redirects=True)
            evento(etapa,url,r.status_code, f'intento={intento+1}; bytes={len(r.content)}')
            r.raise_for_status()
            if not r.content: raise ValueError('Respuesta vacía.')
            if destino is not None:
                tipo = destino.suffix.lower()
                if tipo == '.pdf' and not r.content.startswith(b'%PDF-'):
                    raise ValueError('La respuesta no contiene un PDF.')
                if tipo == '.xlsx' and not r.content.startswith(b'PK\x03\x04'):
                    raise ValueError('La respuesta no contiene un XLSX.')
                if tipo == '.json': json.loads(r.content.decode('utf-8-sig'))
                destino.write_bytes(r.content)
            return r
        except (requests.RequestException,ValueError) as exc:
            ultimo = exc
            if intento + 1 < CONFIG['intentos']:time.sleep(min(3*(intento+1),8))
    raise RuntimeError(f'No se pudo obtener el archivo oficial. URL={url}; error={ultimo}')

def enlaces_bs(url, html):
    sopa = BeautifulSoup(html,'html.parser')
    hallados = []
    for a in sopa.select('a[href]'):
        destino = urljoin(url,a.get('href',''))
        texto = a.get_text(' ',strip=True)
        if re.search(r'(?i)(\.pdf|\.xlsx)(\?|$)|/uploads/document/file/|/storage/20\d\d/',destino):
            hallados.append({'pagina':url,'texto':texto,'url':destino,'via':'BeautifulSoup'})
    return hallados

def enlaces_selenium(url):
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    opts = webdriver.ChromeOptions()
    opts.add_argument('--headless=new')
    opts.add_argument('--disable-gpu')
    opts.add_argument('--window-size=1400,980')
    opts.add_argument('--no-sandbox')
    nav = webdriver.Chrome(options=opts)
    try:
        nav.set_page_load_timeout(55)
        nav.get(url)
        WebDriverWait(nav,20).until(lambda d: d.find_elements(By.CSS_SELECTOR,'a[href]'))
        time.sleep(2)
        resultado = []
        for a in nav.find_elements(By.CSS_SELECTOR,'a[href]'):
            destino = a.get_attribute('href') or ''
            if re.search(r'(?i)(\.pdf|\.xlsx)(\?|$)|/uploads/document/file/|/storage/20\d\d/',destino):
                resultado.append({'pagina':url,'texto':a.text.strip(),'url':destino,'via':'Selenium'})
        return resultado
    finally:
        nav.quit()

CATALOGO_HTML = []
for periodo,pagina in PAGINAS_APN.items():
    encontrados = []
    # Selenium obtiene el contenido que pudiera depender de JavaScript.
    if CONFIG['usar_selenium'] and os.environ.get('HHI_PRUEBA_SIN_NAVEGADOR') != '1':
        try:
            if robots_permite(pagina): encontrados = enlaces_selenium(pagina)
            evento('pagina_selenium',pagina,'correcto',len(encontrados))
        except Exception as exc:
            evento('pagina_selenium',pagina,'no_disponible',repr(exc))
    try:
        r = solicitar(pagina,'pagina_html')
        encontrados += enlaces_bs(pagina,r.text)
    except Exception as exc:
        evento('pagina_html',pagina,'no_disponible',repr(exc))
    CATALOGO_HTML.extend(encontrados)
CATALOGO_HTML = pd.DataFrame(CATALOGO_HTML,columns=['pagina','texto','url','via']).drop_duplicates()
CATALOGO_HTML.to_csv(CATALOGOS/'enlaces_descubiertos_apn.csv',index=False,encoding='utf-8-sig')
print('Enlaces institucionales encontrados en las páginas:',len(CATALOGO_HTML))

In [ ]:
DESCARGAS = []
ENLACES_HALLADOS = {urlparse(u).path: u for u in CATALOGO_HTML.url.dropna().tolist()}
for f in FUENTES_APN.itertuples(index=False):
    periodo = str(f.periodo) if pd.notna(f.periodo) else ''
    if not periodo or periodo == 'nan': periodo = 'historico'
    extension = '.pdf' if f.tipo == 'contenedores_pdf' else '.xlsx'
    nombre = f'APN_{periodo}_{f.tipo}{extension}'
    destino = ORIG_APN / nombre
    if destino.exists():
        digest = hashlib.sha256(destino.read_bytes()).hexdigest()
        if digest != f.sha256: destino.unlink()  # Se reemplaza únicamente una copia incompatible.
    url_descarga = ENLACES_HALLADOS.get(urlparse(f.url).path, f.url)
    if urlparse(url_descarga).netloc not in ('cdn.www.gob.pe','piep.apn.gob.pe','www.gob.pe'):
        raise ValueError('El documento descubierto no pertenece a un portal oficial: ' + url_descarga)
    if not destino.exists():
        solicitar(url_descarga, 'documento_apn', destino)
    digest = hashlib.sha256(destino.read_bytes()).hexdigest()
    coincide = digest == f.sha256
    DESCARGAS.append({'periodo':periodo,'tipo':f.tipo,'archivo':nombre,'url':url_descarga,'url_catalogada':f.url,
                      'sha256_descargado':digest,'sha256_referencia':f.sha256,
                      'coincide_con_version_del_articulo':coincide,'bytes':destino.stat().st_size})
    print(f'{len(DESCARGAS):02d}/56 {periodo:>9} {f.tipo:<18} SHA-256: {"igual" if coincide else "actualizado"}')
DESCARGAS = pd.DataFrame(DESCARGAS)
DESCARGAS.to_csv(CATALOGOS/'documentos_apn_descargados.csv',index=False,encoding='utf-8-sig')
if CONFIG['comprobar_sha_archivado'] and not DESCARGAS.coincide_con_version_del_articulo.all():
    distintos=DESCARGAS.loc[~DESCARGAS.coincide_con_version_del_articulo,['periodo','archivo','url']]
    raise ValueError('La institución ha publicado archivos diferentes de la versión del artículo. Revisar antes de estimar:\n'+distintos.to_string(index=False))
print('Documentos APN descargados:',len(DESCARGAS))

## 5. Consulta de la API del Banco Central de Reserva del Perú

Se consultan cuatro series del BCRP por su código oficial y su intervalo mensual. Las respuestas se conservan en formato JSON y posteriormente se interpretan como series fechadas. Este procedimiento permite distinguir los volúmenes de exportaciones e importaciones de sus valores FOB, ya que el modelo econométrico utiliza tasas de variación del volumen comercial.

In [ ]:
meses_bcrp = {v.lower():i+1 for i,v in enumerate('Ene Feb Mar Abr May Jun Jul Ago Sep Oct Nov Dic'.split())}
series_bcrp = []
for codigo,nombre in CONFIG['codigos_bcrp'].items():
    url = f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{codigo}/json/{CONFIG["inicio"]}/{CONFIG["fin"]}/esp'
    destino = ORIG_BCRP / (codigo+'.json')
    if not destino.exists(): solicitar(url,'api_bcrp',destino)
    contenido = json.loads(destino.read_text(encoding='utf-8-sig'))
    registros = []
    for p in contenido.get('periods',[]):
        partes = str(p['name']).split('.')
        if len(partes)!=2:raise ValueError(f'Periodo BCRP inesperado: {p["name"]}')
        mes = meses_bcrp[partes[0].strip()[:3].lower()]
        valor = pd.to_numeric(str(p['values'][0]).replace(',',''),errors='raise')
        registros.append({'mes':f'{int(partes[1]):04d}-{mes:02d}',nombre:float(valor)})
    tabla=pd.DataFrame(registros).set_index('mes')
    assert len(tabla)==139 and tabla.index.is_unique and tabla.notna().all().all(),codigo
    series_bcrp.append(tabla)
BCRP = pd.concat(series_bcrp,axis=1).sort_index().reset_index()
assert len(BCRP)==139 and BCRP.mes.is_unique
BCRP.to_csv(PROCESADOS/'BCRP_MENSUAL.csv',index=False,encoding='utf-8-sig')
print('Series del BCRP recuperadas y conciliadas:',len(BCRP),'meses.')
display(BCRP.head(3))

## 6. Lectura del Excel histórico de la APN, 2015–2021

El libro histórico de la APN organiza los meses en columnas y diferencia el Callao entre APM Terminals y DP World. Se recuperan las observaciones mensuales de ambos terminales y Paita. Para comprobar la selección de las celdas, la suma de APM y DP World debe coincidir con el subtotal publicado para el Callao en cada mes.

In [ ]:
PERIODOS = pd.period_range(CONFIG['inicio'], CONFIG['fin'], freq='M')
RAIZ = ORIG_APN
HIST = next(RAIZ.glob('APN_historico_historico_xlsx.xlsx'))
URLS = DESCARGAS
rows = []
checks = []
operaciones = []

def file_info(archivo):
    return (archivo.name, hashlib.sha256(archivo.read_bytes()).hexdigest())

def source_url(nombre):
    
    r = DESCARGAS.loc[DESCARGAS.archivo==nombre,'url']
    return str(r.iloc[0]) if len(r) else ''

def put(mes, terminal, valor, archivo, metodo, calidad='comprobacion_automatica', nota=''):
    assert pd.Period(mes, freq='M') in PERIODOS
    assert isinstance(valor, (int, float, np.integer, np.floating)) and 0 <= valor < 2000000
    nombre, sha = file_info(archivo)
    rows.append(dict(mes=mes, terminal=terminal, teu=int(round(valor)), metodo=metodo, validacion=calidad, nota_revision=nota, archivo_original=nombre, sha256_fuente=sha, url_fuente=source_url(nombre)))
libro = load_workbook(HIST, read_only=False, data_only=True)
hoja = libro['TEUs']
for columna in range(4, hoja.max_column + 1):
    fecha = hoja.cell(6, columna).value
    if not isinstance(fecha, datetime) or not 2015 <= fecha.year <= 2021:
        continue
    mes = fecha.strftime('%Y-%m')
    apm = hoja.cell(19, columna).value
    dpw = hoja.cell(20, columna).value
    paita = hoja.cell(13, columna).value
    subtotal_callao = hoja.cell(18, columna).value
    assert all((v is not None for v in (apm, dpw, paita, subtotal_callao)))
    # La suma de los dos terminales debe coincidir con el subtotal del Callao.
    assert int(apm) + int(dpw) == int(subtotal_callao), mes
    for terminal, valor in [('APMTC_CALLAO', apm), ('DPWC_CALLAO', dpw), ('TPE_PAITA', paita)]:
        put(mes, terminal, valor, HIST, 'Excel_historico_TEUs_filas_6_19_20_13')
    checks.append((mes, 'consistencia_callao', int(subtotal_callao) - int(apm) - int(dpw)))
libro.close()
assert len(rows) == 84 * 3
print('Excel histórico: recuperados 84 meses y', len(rows), 'registros de terminal.')

## 7. Procesamiento de los resúmenes Excel, 2022–2023

Las publicaciones mensuales utilizan la pestaña de tipo de carga. El dato del mes corriente se encuentra en la columna C, mientras otras columnas representan acumulados o periodos de comparación. Se verifica el encabezado TEU y la igualdad entre los dos terminales del Callao y su subtotal. Las operaciones se conservan en una tabla adicional para documentar el procesamiento.

In [ ]:
archivos_resumen = sorted(RAIZ.glob('APN_20??-??_resumen_xlsx.xlsx'))
assert len(archivos_resumen) == 24
for archivo in archivos_resumen:
    mes = re.search('APN_(\\d{4}-\\d{2})_resumen', archivo.name).group(1)
    libro = load_workbook(archivo, read_only=False, data_only=True)
    hoja = libro['TP Uso Público - Tipo de carga']
    assert 'TEU' in str(hoja['C16'].value).upper(), archivo.name
    valores = {'APMTC_CALLAO': hoja['C29'].value, 'DPWC_CALLAO': hoja['C30'].value, 'TPE_PAITA': hoja['C25'].value}
    assert all((isinstance(v, (int, float)) for v in valores.values()))
    assert int(valores['APMTC_CALLAO']) + int(valores['DPWC_CALLAO']) == int(hoja['C28'].value)
    for terminal, valor in valores.items():
        put(mes, terminal, valor, archivo, 'Excel_resumen_columna_C_del_mes_actual')
    checks.append((mes, 'consistencia_callao', 0))
    for pestaña, tipo in [('TP Uso Público - DESCARGA', 'descarga'), ('TP Uso Público - EMBARQUE', 'embarque'), ('TP Uso Público - TRANSBORDO', 'transbordo'), ('TP Uso Público - REESTIBA', 'reestiba')]:
        op = libro[pestaña]
        for terminal, fila in [('APMTC_CALLAO', 29), ('DPWC_CALLAO', 30), ('TPE_PAITA', 25)]:
            valor = op.cell(fila, 3).value
            operaciones.append(dict(mes=mes, terminal=terminal, operacion=tipo, teu=int(valor) if pd.notna(valor) else None, archivo_original=archivo.name))
    libro.close()
assert len(rows) == (84 + 24) * 3
pd.DataFrame(operaciones).to_csv(PROCESADOS / 'APN_OPERACIONES_2022_2023.csv', index=False, encoding='utf-8-sig')
print('Excel 2022–2023: 24 archivos procesados; registros acumulados:', len(rows))

## 8. Extracción de TEU desde reportes PDF, 2024–2026

Los informes de contenedores muestran un ranking de terminales. Cuando el PDF contiene texto, se identifican las etiquetas y los TEU de la misma fila mediante coordenadas de página. En los cuatro informes de imagen del periodo de transición se conserva el documento descargado y se registra la imposibilidad de recuperar automáticamente la tabla con este procedimiento. Dichos meses permanecen fuera del HHI conjunto, tal como se delimitó la muestra. No se incorporan valores copiados manualmente ni cifras precargadas.

In [ ]:
TERM_MAP = {'DPWC':'DPWC_CALLAO','APMTC':'APMTC_CALLAO','TPE':'TPE_PAITA','COSCO':'COSCO_CHANCAY'}
PDFS = sorted(RAIZ.glob('APN_20??-??_contenedores_pdf.pdf'))
assert len(PDFS)==31
INCIDENCIAS_PDF = []
for archivo in PDFS:
    mes=re.search(r'APN_(\d{4}-\d{2})_',archivo.name).group(1)
    with fitz.open(archivo) as documento:
        pagina = documento[0]
        palabras = pagina.get_text('words')
        ancho = pagina.rect.width
    if len(palabras)<40:
        INCIDENCIAS_PDF.append({'mes':mes,'archivo':archivo.name,'motivo':'PDF de imagen sin tabla textual'})
        if not '2024-11'<=mes<'2025-06':
            raise ValueError('Es necesario obtener una fuente tabular verificable para '+mes)
        continue
    encontrados = []
    for etiqueta,terminal in TERM_MAP.items():
        anclas=[p for p in palabras if str(p[4]).upper()==etiqueta and p[0]>.7*ancho and 105<p[1]<340]
        numeros=[p for p in palabras if re.fullmatch(r'\d{1,3}(?:,\d{3})+',str(p[4])) and p[0]>.78*ancho and 105<p[1]<340]
        candidatos=[(abs(a[1]-v[1]),v) for a in anclas for v in numeros if abs(a[1]-v[1])<14 and v[0]>a[0]]
        if candidatos:
            valor = min(candidatos,key=lambda t:t[0])[1]
            put(mes,terminal,int(valor[4].replace(',','')),archivo,'PDF_texto_ranking')
            encontrados.append(terminal)
    faltan=set(['DPWC_CALLAO','APMTC_CALLAO','TPE_PAITA'])-set(encontrados)
    if faltan:
        INCIDENCIAS_PDF.append({'mes':mes,'archivo':archivo.name,'motivo':'No encontrados: '+', '.join(sorted(faltan))})
        if not '2024-11'<=mes<'2025-06':
            raise ValueError(f'No se pudo leer el documento {archivo.name}: {faltan}')
APN = pd.DataFrame(rows).sort_values(['mes','terminal']).reset_index(drop=True)
assert not APN.duplicated(['mes','terminal']).any()
APN.to_csv(PROCESADOS/'APN_DATOS_TRAZABLES.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(INCIDENCIAS_PDF,columns=['mes','archivo','motivo']).to_csv(AUDITORIA/'incidencias_pdf.csv',index=False,encoding='utf-8-sig')
print('Reportes PDF consultados:',len(PDFS),'| Incidencias en transición:',len(INCIDENCIAS_PDF))
display(APN.groupby('metodo').size().rename('registros').to_frame())

## 9. Integración mensual y control de procedencia

Se reúnen los tráficos portuarios y las series del BCRP mediante la llave año–mes. El calendario conserva todos los meses del estudio, incluso cuando existe un PDF que no admite extracción de texto. Se exige que las variables requeridas por el HHI estén disponibles fuera de la transición. Cada registro portuario mantiene la referencia al documento del que procede.

In [ ]:
PERIODOS = pd.period_range(CONFIG['inicio'],CONFIG['fin'],freq='M')
wide = APN.pivot(index='mes',columns='terminal',values='teu').reindex([str(x) for x in PERIODOS])
for columna in ['DPWC_CALLAO','APMTC_CALLAO','TPE_PAITA','COSCO_CHANCAY']:
    if columna not in wide:wide[columna] = np.nan
validos = (wide.index < CONFIG['transicion']) | (wide.index >= CONFIG['comercial'])
assert wide.loc[validos,['DPWC_CALLAO','APMTC_CALLAO','TPE_PAITA']].notna().all().all()
assert wide.loc[wide.index>=CONFIG['comercial'],'COSCO_CHANCAY'].notna().all()
wide['CALLAO_TEU'] = wide['DPWC_CALLAO']+wide['APMTC_CALLAO']
wide = wide.reset_index()
wide['fecha'] = pd.to_datetime(wide.mes+'-01')
wide['indicador_comercial'] = (wide.mes>=CONFIG['comercial']).astype(int)
wide['indicador_pruebas_transicion'] = ((wide.mes>=CONFIG['transicion'])&(wide.mes<CONFIG['comercial'])).astype(int)
wide['pandemia_2020_2021'] = ((wide.mes>='2020-03')&(wide.mes<='2021-12')).astype(int)
wide['tendencia_mensual'] = np.arange(1,len(wide)+1)
wide['mes_calendario'] = wide.fecha.dt.month
mes_post = wide.loc[wide.indicador_comercial==1,'tendencia_mensual'].iloc[0]
wide['post_comercial_meses'] = np.where(wide.indicador_comercial==1,wide.tendencia_mensual-mes_post+1,0)
base = wide.merge(BCRP,on='mes',how='left',validate='one_to_one')
base['serie_apn_completa'] = base[['DPWC_CALLAO','APMTC_CALLAO']].notna().all(axis=1)
base['serie_bcrp_completa'] = base[list(CONFIG['codigos_bcrp'].values())].notna().all(axis=1)
base['apta_para_modelo'] = (base.serie_apn_completa & base.serie_bcrp_completa
                           & base.indicador_pruebas_transicion.eq(0)
                           & (base.indicador_comercial.eq(0) | base.COSCO_CHANCAY.notna()))
assert int(base.apta_para_modelo.sum()) == 132
assert len(base)==139 and base.loc[validos,'apta_para_modelo'].all()
base.to_csv(PROCESADOS/'BASE_MENSUAL_INTEGRADA_2015_2026.csv',index=False,encoding='utf-8-sig')
print('Calendario completo:',len(base),'meses. Meses aptos para el modelo:',int(base.apta_para_modelo.sum()))
display(base[['mes','APMTC_CALLAO','DPWC_CALLAO','COSCO_CHANCAY']].tail(5))

## 10. Cálculo del HHI y sus medidas complementarias

La participación de un terminal se calcula dividiendo su tráfico mensual entre la suma de los terminales del conjunto estudiado. El HHI pondera al cuadrado esas participaciones, por lo que concede más peso a los operadores de mayor tamaño. Se incorporan también el número equivalente de participantes y un HHI normalizado para examinar la desigualdad de las cuotas respecto de un reparto simétrico.

$$
s_{it}=\frac{Q_{it}}{\sum_{j=1}^{N_t}Q_{jt}},\qquad
\operatorname{HHI}_t=10\,000\sum_{i=1}^{N_t}s_{it}^{2}.
$$

$$
N_t^{\mathrm{eq}}=\frac{10\,000}{\operatorname{HHI}_t},\qquad
\operatorname{HHI}_t^{\mathrm{norm}}=\frac{\operatorname{HHI}_t/10\,000-1/N_t}{1-1/N_t}.
$$

In [ ]:
d = base.set_index('fecha').sort_index().asfreq('MS').copy()
pre = d.index < pd.Timestamp(CONFIG['transicion']+'-01')
post = d.index >= pd.Timestamp(CONFIG['comercial']+'-01')
transicion = ~(pre|post)
ch = d.COSCO_CHANCAY.copy()
ch.loc[pre] = 0
ch.loc[transicion] = np.nan
sistema = d.CALLAO_TEU+ch
cuotas = pd.DataFrame({'APM':d.APMTC_CALLAO/sistema,'DPW':d.DPWC_CALLAO/sistema,'Chancay':ch/sistema})
d['r_chancay'] = ch/sistema
d['hhi_inc'] = 10000*((d.APMTC_CALLAO/d.CALLAO_TEU)**2+(d.DPWC_CALLAO/d.CALLAO_TEU)**2)
d['hhi_sys'] = 10000*cuotas.pow(2).sum(axis=1,min_count=3)
d['hhi_norm'] = np.where(pre,2*(d.hhi_inc/10000)-1,(d.hhi_sys/10000-1/3)/(1-1/3))
d['neff'] = 10000/d.hhi_sys
d['hhi_including_paita'] = 10000*(d[['APMTC_CALLAO','DPWC_CALLAO','TPE_PAITA']].div(sistema+d.TPE_PAITA,axis=0).pow(2).sum(axis=1,min_count=3)+(ch/(sistema+d.TPE_PAITA))**2)
d['inc_changeable'] = d.hhi_inc.copy()
d.loc[transicion,'inc_changeable'] = np.nan
assert len(d)==139 and d.hhi_sys.notna().sum()==132 and d.loc[transicion,'hhi_sys'].isna().all()
assert d.apta_para_modelo.eq(d.hhi_sys.notna()).all(), 'La clasificación debe coincidir con el HHI disponible.'
d[['APMTC_CALLAO','DPWC_CALLAO','COSCO_CHANCAY','r_chancay','hhi_inc','hhi_sys','hhi_norm','neff','hhi_including_paita']].to_csv(PROCESADOS/'INDICADORES_HHI_MENSUALES.csv',encoding='utf-8-sig',index_label='fecha')
print('HHI construido con 132 meses válidos. Siete meses de transición sin HHI.')
display(d[['hhi_inc','hhi_sys','hhi_norm','neff']].loc['2024-09':'2025-08'])

## 11. Descomposición de la variación de concentración

Sea $r_t$ la participación de Chancay y $C_t$ el HHI de los dos terminales del Callao, con participaciones normalizadas dentro del propio Callao. La identidad siguiente permite separar la participación de Chancay de los cambios internos del reparto entre APM y DP World.

$$
\operatorname{HHI}_t^{\mathrm{conjunto}}=(1-r_t)^2C_t+10\,000\,r_t^2.
$$

Para cada mes $m$ de junio a octubre se compara 2024 con el mismo mes de 2025. La descomposición contable utiliza

$$
\begin{aligned}
E_m &= f(r_{1m},C_{0m})-f(0,C_{0m}),\\
R_m &= f(r_{1m},C_{1m})-f(r_{1m},C_{0m}),\\
\Delta\operatorname{HHI}_m &= E_m+R_m,
\end{aligned}
\qquad f(r,C)=(1-r)^2C+10\,000r^2.
$$

Como comprobación, la descomposición de Shapley promedia los dos órdenes de actualización.

$$
\begin{aligned}
E_m^S&=\tfrac12\bigl[f(r_{1m},C_{0m})-f(0,C_{0m})+f(r_{1m},C_{1m})-f(0,C_{1m})\bigr],\\
R_m^S&=\Delta\operatorname{HHI}_m-E_m^S.
\end{aligned}
$$

In [ ]:
def paired(d):
    out = []
    f = lambda r, h: (1 - r) ** 2 * h + 10000 * r * r
    # Cada mes de 2025 se compara con el mismo mes de 2024.
    for month in CONFIG['comparacion_meses']:
        a = d.loc[pd.Timestamp(2024, month, 1)]
        b = d.loc[pd.Timestamp(2025, month, 1)]
        r = b.r_chancay
        h0 = a.hhi_inc
        h1 = b.hhi_inc
        j = f(r, h1)
        k = f(r, h0)
        out.append(dict(mes_num=month, mes_2024=f"{2024}-{month:02d}", mes_2025=f"{2025}-{month:02d}", HHI_2024=h0, HHI_2025=j, cambio=j - h0, cuota_chancay=100 * r, HHI_callao_2024=h0, HHI_callao_2025=h1, cambio_interno_callao=h1 - h0, componente_entrada=k - h0, componente_interno=j - k, shapley_entrada=0.5 * (f(r, h0) - h0 + f(r, h1) - h1), shapley_interno=0.5 * (h1 - h0 + f(r, h1) - f(r, h0)), HHI_normalizado_2024=a.hhi_norm, HHI_normalizado_2025=b.hhi_norm, participantes_efectivos_2024=a.neff, participantes_efectivos_2025=b.neff))
    q = pd.DataFrame(out)
    assert np.max(np.abs(q.cambio - (q.componente_entrada + q.componente_interno))) < 1e-07
    assert np.max(np.abs(q.cambio - (q.shapley_entrada + q.shapley_interno))) < 1e-07
    return q
q = paired(d)
assert np.allclose(q.cambio, q.componente_entrada + q.componente_interno, atol=1e-07)
assert np.allclose(q.cambio, q.shapley_entrada + q.shapley_interno, atol=1e-07)
q.to_csv(RESULTADOS / 'descomposicion_pareada_junio_octubre.csv', index=False, encoding='utf-8-sig')
fraccion_entrada = q.componente_entrada.sum() / q.cambio.sum()
fraccion_shapley = q.shapley_entrada.sum() / q.cambio.sum()
print('Caída media HHI:', round(q.cambio.mean(), 2), 'puntos')
print('Componente asociado a Chancay:', round(fraccion_entrada * 100, 2), '%')
print('Asignación de Shapley:', round(fraccion_shapley * 100, 2), '%')
display(q[['mes_2024', 'mes_2025', 'HHI_2024', 'HHI_2025', 'componente_entrada', 'componente_interno']].round(2))

## 12. Concordancia entre las publicaciones PDF y Excel de 2025

Se comparan los volúmenes de APM Terminals Callao, DP World Callao y Chancay declarados en los reportes PDF y en los resúmenes Excel de la APN. Cada comparación conserva el mes, el terminal y la dirección del documento oficial.

La coincidencia de las cifras entre dos formatos representa una comprobación documental. No se utiliza para estimar nuevamente el mismo modelo, pues ambos archivos contienen el mismo tráfico de contenedores.

In [ ]:
MESES_PALABRAS = {'enero':1,'febrero':2,'marzo':3,'abril':4,'mayo':5,'junio':6,
                  'julio':7,'agosto':8,'septiembre':9,'setiembre':9,'octubre':10,
                  'noviembre':11,'diciembre':12}
fuentes_complementarias = []
for fila in CATALOGO_HTML.itertuples(index=False):
    if '/6573509-' not in fila.url and '2025' not in fila.url.lower():
        continue
    texto = unquote(fila.url + ' ' + fila.texto).lower()
    if not re.search(r'\.xlsx(?:\?|$)', fila.url, re.I):
        continue
    if 'resumen' not in texto or 'carga' not in texto:
        continue
    for palabra, numero in MESES_PALABRAS.items():
        if re.search(r'\b' + palabra + r'\b', texto):
            fuentes_complementarias.append({'mes': f'2025-{numero:02d}',
                                            'url': fila.url, 'pagina': fila.pagina})
            break
CAT_COMPLEMENTARIO = pd.DataFrame(fuentes_complementarias,
                                  columns=['mes','url','pagina']).drop_duplicates('mes')
CAT_COMPLEMENTARIO.to_csv(CATALOGOS/'fuentes_complementarias_2025.csv',
                          index=False, encoding='utf-8-sig')

def extraer_teu_excel(ruta):
    libro = load_workbook(ruta, read_only=True, data_only=True)
    try:
        if 'TP Uso Público - Tipo de carga' not in libro.sheetnames:
            return None
        hoja = libro['TP Uso Público - Tipo de carga']
        valores = {}
        for fila in hoja.iter_rows(min_row=16, max_row=min(hoja.max_row,125),
                                   max_col=4, values_only=True):
            etiqueta = ' '.join(str(v or '') for v in fila[:2]).upper()
            cantidad = fila[2]
            if not isinstance(cantidad,(int,float)) or cantidad < 0:
                continue
            if 'APM' in etiqueta:
                valores['APMTC_CALLAO'] = int(cantidad)
            elif 'DP WORLD' in etiqueta or 'DPWC' in etiqueta:
                valores['DPWC_CALLAO'] = int(cantidad)
            elif 'COSCO' in etiqueta or 'CHANCAY' in etiqueta:
                valores['COSCO_CHANCAY'] = int(cantidad)
        return valores if len(valores)==3 else None
    finally:
        libro.close()

comparaciones = []
incidencias_complementarias = []
terminales_cotejados = ['APMTC_CALLAO','DPWC_CALLAO','COSCO_CHANCAY']
for fuente in CAT_COMPLEMENTARIO.itertuples(index=False):
    if not ('2025-06' <= fuente.mes <= '2025-12'):
        continue
    archivo = ORIG_APN/f'APN_{fuente.mes}_resumen_complementario.xlsx'
    try:
        if not archivo.exists():
            solicitar(fuente.url,'xlsx_complementario',archivo)
        valores_excel = extraer_teu_excel(archivo)
        if valores_excel is None:
            incidencias_complementarias.append({'mes': fuente.mes,
                                                'observacion': 'No se identificó la tabla de TEU'})
            continue
        for terminal in terminales_cotejados:
            fila_pdf = APN.loc[(APN.mes==fuente.mes) & (APN.terminal==terminal)]
            if len(fila_pdf)!=1:
                raise ValueError(f'Sin dato PDF único para {fuente.mes}, {terminal}')
            teu_pdf = int(fila_pdf.iloc[0].teu)
            teu_excel = int(valores_excel[terminal])
            comparaciones.append({
                'mes': fuente.mes, 'terminal': terminal, 'teu_pdf': teu_pdf,
                'teu_excel': teu_excel, 'diferencia': teu_excel-teu_pdf,
                'coinciden': teu_excel==teu_pdf,
                'url_pdf': fila_pdf.iloc[0].url_fuente, 'url_excel': fuente.url
            })
    except Exception as exc:
        incidencias_complementarias.append({'mes': fuente.mes, 'observacion': str(exc)})

concordancia = pd.DataFrame(comparaciones,
    columns=['mes','terminal','teu_pdf','teu_excel','diferencia','coinciden','url_pdf','url_excel'])
concordancia.to_csv(CATALOGOS/'concordancia_PDF_Excel_2025.csv',
                   index=False,encoding='utf-8-sig')
pd.DataFrame(incidencias_complementarias, columns=['mes','observacion']).to_csv(
    AUDITORIA/'incidencias_excel_complementario.csv',index=False,encoding='utf-8-sig')
if len(concordancia):
    if not concordancia.coinciden.all():
        raise ValueError('Se hallaron diferencias PDF-Excel. Revisar concordancia_PDF_Excel_2025.csv.')
    print('Comparaciones coincidentes:',len(concordancia),
          '| meses cotejados:',concordancia.mes.nunique())
else:
    print('No se encontraron Excel complementarios comparables; el análisis principal continúa con los PDF oficiales.')


## 13. Modelo econométrico principal

Se estima la asociación mensual entre la etapa comercial de Chancay y la concentración del tráfico mediante una regresión con tendencia, indicadores mensuales, dos controles temporales asociados a la pandemia y tasas interanuales de volumen de exportaciones e importaciones. La perturbación sigue un proceso AR(2).

$$
\begin{aligned}
\operatorname{HHI}_t={}&\alpha+\delta D_t+\beta\tau_t+
\sum_{m=2}^{12}\gamma_mM_{mt}\\
&+\theta_1P_t+\theta_2L_t+\lambda_Xg_{X,t}+\lambda_Mg_{M,t}+u_t,\\
u_t={}&\rho_1u_{t-1}+\rho_2u_{t-2}+\varepsilon_t.
\end{aligned}
$$

El indicador $D_t$ toma el valor uno desde junio de 2025. La tendencia $\tau_t$ mide el tiempo transcurrido en años desde enero de 2015, y los indicadores $M_{mt}$ permiten controlar la estacionalidad. La regresión se estima mediante máxima verosimilitud con `SARIMAX(2,0,0)`. El HHI se divide entre mil para el ajuste numérico y después se reexpresan los resultados en puntos. Los siete meses de transición permanecen en el calendario como valores ausentes de la variable dependiente.

In [ ]:
def exog_design(d, controls=True):
    X = pd.DataFrame({'const': np.ones(len(d)), 'trend': np.arange(len(d)) / 12.0, 'post': (d.index >= pd.Timestamp(CONFIG['comercial'] + '-01')).astype(float)}, index=d.index)
    for m in range(2, 13):
        X[f'mes{m}'] = (d.index.month == m).astype(float)
    X['covid'] = ((d.index >= CONFIG['meses_covid'][0] + '-01') & (d.index <= CONFIG['meses_covid'][1] + '-01')).astype(float)
    X['lockdown'] = ((d.index >= CONFIG['meses_covid'][0] + '-01') & (d.index <= CONFIG['meses_confinamiento'][1] + '-01')).astype(float)
    if controls:
        X['exp_yoy'] = d.var_interanual_volumen_exportado_pct / 100
        X['imp_yoy'] = d.var_interanual_volumen_importado_pct / 100
    return X
X = exog_design(d)
assert len(X) == 139 and (not X.isna().any().any())
assert int(X.post.sum()) == 14
print('Regresores incluidos en el modelo:')
print(', '.join(X.columns))
print('Etapa comercial:', int(X.post.sum()), 'meses; observaciones válidas:', int(d.hhi_sys.notna().sum()))

## 14. Estimación principal y especificaciones de contraste

La estimación principal utiliza errores AR(2). Se presentan siete especificaciones adicionales, entre ellas ARMA(1,1), el modelo sin controles comerciales, dos variantes de MCO con corrección HAC, la concentración del Callao, el alcance ampliado que incorpora Paita y la muestra desde 2021. Los resultados consignan el coeficiente, el intervalo de confianza, la convergencia y el diagnóstico de Ljung–Box que corresponda.

In [ ]:
def regressions(d):
    X = exog_design(d)
    results = []
    global FITS
    FITS = {}
    models = [
        ('HHI conjunto - AR(2)', d.hhi_sys, (2, 0, 0), True, 'AR'), ('HHI conjunto - ARMA(1,1)', d.hhi_sys, (1, 0, 1), True, 'AR'), ('HHI conjunto - AR(2) sin controles comerciales', d.hhi_sys, (2, 0, 0), False, 'AR'), ('HHI conjunto - MCO-HAC', d.hhi_sys, None, True, 'HAC'), ('HHI del Callao - AR(2)', d.inc_changeable, (2, 0, 0), True, 'AR'), ('HHI conjunto con Paita - AR(2)', d.hhi_including_paita, (2, 0, 0), True, 'AR'), ('HHI conjunto - AR(2) periodo 2021-2026', d.hhi_sys.loc[CONFIG['inicio_ventana']:], (2, 0, 0), True, 'AR'), ('HHI conjunto - MCO-HAC con pendiente posterior', d.hhi_sys, None, True, 'HAC_POST_TREND'),
    ]
    for title, y, order, with_controls, mt in models:
        xx = exog_design(d, with_controls).reindex(y.index)
        if mt == 'HAC_POST_TREND':
            xx['post_trend'] = np.where(xx.post == 1, np.arange(len(xx)) - np.flatnonzero(xx.post.to_numpy())[0] + 1, 0) / 12.0
        # La escala en miles favorece la convergencia del estimador dinámico.
        yy = y / 1000
        try:
            if mt.startswith('HAC'):
                ok = yy.notna() & xx.notna().all(axis=1)
                fit = sm.OLS(yy.loc[ok], xx.loc[ok]).fit(cov_type='HAC', cov_kwds={'maxlags': CONFIG['retardo_hac'], 'use_correction': True})
                e = fit.resid
                converged = True
                nobs = int(ok.sum())
            else:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    fit = SARIMAX(yy, exog=xx, order=order, trend='n', enforce_stationarity=True, enforce_invertibility=True).fit(disp=False, maxiter=CONFIG['max_iteraciones'], cov_type='robust')
                converged = bool(fit.mle_retvals.get('converged', False))
                nobs = int(yy.notna().sum())
                ee = pd.Series(fit.filter_results.standardized_forecasts_error[0], index=yy.index)
                e = ee[(ee.index < CONFIG['transicion'] + '-01') & yy.notna()].iloc[12:]
            FITS[title] = fit
            ci = fit.conf_int().loc['post'] * 1000
            lb = acorr_ljungbox(e.dropna(), lags=[CONFIG['retardo_ljungbox']], return_df=True).lb_pvalue.iloc[0]
            results.append(dict(modelo=title, n=nobs, coef_post=float(fit.params.post * 1000), ic95_inferior=float(ci.iloc[0]), ic95_superior=float(ci.iloc[1]), p=float(fit.pvalues.post), convergio=converged, lb12_p=float(lb), coef_pendiente_post=float(fit.params.get('post_trend', np.nan) * 1000), p_pendiente_post=float(fit.pvalues.get('post_trend', np.nan))))
        except Exception as e:
            results.append(dict(modelo=title, error=repr(e)))
    return pd.DataFrame(results)
modelos = regressions(d)
assert len(modelos)==8,modelos.to_string(index=False)
assert modelos.convergio.fillna(False).all(), 'Algún modelo no convergió: ' + modelos.to_string(index=False)
assert np.isfinite(modelos.coef_post).all()
modelos.to_csv(RESULTADOS / 'modelos_y_diagnosticos.csv', index=False, encoding='utf-8-sig')
print('Especificaciones econométricas estimadas:',len(modelos))
display(modelos[['modelo', 'n', 'coef_post', 'ic95_inferior', 'ic95_superior', 'p', 'lb12_p']].round(4))

## 15. Diagnósticos y comprobación de resultados

Se conserva el periodo completo en el modelo dinámico, incluidos los valores faltantes del HHI en la transición. La comparación descriptiva utiliza cinco pares de meses de junio a octubre. Se contrastan la consistencia de la descomposición, las observaciones efectivas del modelo y el sentido de los coeficientes. Los resultados numéricos se obtienen de las fuentes descargadas y no se fijan dentro del código.

In [ ]:
principal = FITS['HHI conjunto - AR(2)']
intervalos = principal.conf_int()
coeficientes = pd.DataFrame({'parametro': principal.params.index, 'coeficiente_modelo_escalado': principal.params.values, 'error_estandar_modelo_escalado': principal.bse.values, 'p': principal.pvalues.values, 'IC_inf_modelo_escalado': intervalos.iloc[:, 0].values, 'IC_sup_modelo_escalado': intervalos.iloc[:, 1].values})
escala = np.where(coeficientes.parametro.isin(['ar.L1', 'ar.L2', 'sigma2']), 1.0, 1000.0)
coeficientes['coeficiente_reportado'] = coeficientes.coeficiente_modelo_escalado * escala
coeficientes['error_estandar_reportado'] = coeficientes.error_estandar_modelo_escalado * escala
coeficientes['IC_inf_reportado'] = coeficientes.IC_inf_modelo_escalado * escala
coeficientes['IC_sup_reportado'] = coeficientes.IC_sup_modelo_escalado * escala
coeficientes.to_csv(RESULTADOS / 'coeficientes_completos_AR2.csv', index=False, encoding='utf-8-sig')
(RESULTADOS / 'resumen_estimacion_principal.txt').write_text(str(principal.summary()), encoding='utf-8')
display(coeficientes[coeficientes.parametro.isin(['const', 'post', 'trend', 'covid', 'lockdown', 'exp_yoy', 'imp_yoy', 'ar.L1', 'ar.L2'])][['parametro', 'coeficiente_reportado', 'error_estandar_reportado', 'p', 'IC_inf_reportado', 'IC_sup_reportado']].round(4))

In [ ]:
principal = FITS['HHI conjunto - AR(2)']
resumen = {
 'meses_calendario':int(len(d)),
 'HHI_meses_validos':int(d.hhi_sys.notna().sum()),
 'HHI_2024_junio_octubre':float(q.HHI_2024.mean()),
 'HHI_2025_junio_octubre':float(q.HHI_2025.mean()),
 'cambio_puntos':float(q.cambio.mean()),
 'cambio_pct':float(100*(q.HHI_2025.mean()/q.HHI_2024.mean()-1)),
 'componente_entrada_pct':float(100*q.componente_entrada.sum()/q.cambio.sum()),
 'componente_callao_pct':float(100*q.componente_interno.sum()/q.cambio.sum()),
 'shapley_entrada_pct':float(100*q.shapley_entrada.sum()/q.cambio.sum()),
 'coeficiente_AR2':float(modelos.iloc[0].coef_post),
}
assert len(d)==139 and resumen['HHI_meses_validos']==132
assert abs(q.cambio-(q.componente_entrada+q.componente_interno)).max()<1e-7
assert modelos.convergio.fillna(False).all(),modelos.to_string(index=False)
(RESULTADOS/'resumen_economico.json').write_text(json.dumps(resumen,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame([resumen]).T.rename(columns={0:'Valor'}))

## 16. Figuras de la investigación

Las figuras se generan a partir de la base mensual y de las estimaciones ejecutadas. Se presenta la evolución del HHI conjunto y del Callao, las cuotas de tráfico, la contribución de cada componente a la desconcentración, los coeficientes de las especificaciones disponibles y el HHI normalizado. Los archivos se exportan en PNG y PDF vectorial.

In [ ]:
import matplotlib.dates as mdates
plt.rcParams.update({'font.size': 10, 'axes.labelsize': 10, 'legend.fontsize': 9, 'xtick.labelsize': 9, 'ytick.labelsize': 9, 'pdf.fonttype': 42, 'figure.dpi': 120})
meses = ['ene.', 'feb.', 'mar.', 'abr.', 'may.', 'jun.', 'jul.', 'ago.', 'sep.', 'oct.', 'nov.', 'dic.']

def figbase(height=3.4):
    fig, ax = plt.subplots(figsize=(6.5, height))
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.2)
    ax.set_axisbelow(True)
    return (fig, ax)

def save(fig, n):
    fig.tight_layout(pad=1.0)
    for ext in ['pdf', 'png']:
        fig.savefig(FIGURAS / f'figura_{n:02d}.{ext}', dpi=300, bbox_inches='tight', metadata={'Creator': 'Python / Matplotlib'} if ext == 'pdf' else None)
    plt.close(fig)

def fechas(ax, years=True):
    ax.xaxis.set_major_locator(mdates.YearLocator(2 if years else 1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.set_xlabel('Mes')
fig, ax = figbase(3.5)
ax.plot(d.index, d.hhi_sys, label='Conjunto Callao–Chancay', linewidth=1.8)
ax.plot(d.index, d.hhi_inc, label='APM y DP World, cuotas entre ambos', linestyle='--', linewidth=1.15, alpha=0.85)
ax.axvspan(pd.Timestamp(CONFIG['transicion']), pd.Timestamp(CONFIG['comercial']), alpha=0.09)
ax.axvline(pd.Timestamp(CONFIG['comercial']), linestyle=':', linewidth=1)
ax.set_ylabel('HHI (puntos, escala 0–10 000)')
ax.set_ylim(3900, 8100)
fechas(ax)
ax.legend(loc='upper right', frameon=False)
save(fig, 1)
fig, ax = figbase(3.4)
recent = d.loc[CONFIG['inicio_ventana']:]
trafico_conjunto = recent.CALLAO_TEU / (1 - recent.r_chancay)
ax.plot(recent.index, 100 * recent.DPWC_CALLAO / trafico_conjunto, label='DP World Callao', linewidth=1.7)
ax.plot(recent.index, 100 * recent.APMTC_CALLAO / trafico_conjunto, label='APM Terminals Callao', linewidth=1.7)
ax.plot(recent.index, 100 * recent.r_chancay, label='Chancay', linewidth=1.7)
ax.axvspan(pd.Timestamp(CONFIG['transicion']), pd.Timestamp(CONFIG['comercial']), alpha=0.09)
ax.set_ylim(0, 75)
ax.set_ylabel('Participación en el tráfico conjunto (%)')
fechas(ax, False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=3, frameon=False)
save(fig, 2)
fig, ax = figbase(2.8)
vals = [-q.componente_entrada.mean(), -q.componente_interno.mean()]
ax.barh(['Cuota de Chancay', 'Reparto APM–DP World'], vals, height=0.46)
for i, (v, sh) in enumerate(zip(vals, [resumen['componente_entrada_pct']/100, resumen['componente_callao_pct']/100])):
    ax.text(v + 15, i, f'{v:.1f} ({100 * sh:.1f} %)'.replace('.', ','), va='center', ha='left', fontsize=10)
ax.set_xlim(0, 1250)
ax.invert_yaxis()
ax.set_xlabel('Contribución a la caída del HHI (puntos)')
ax.grid(False)
save(fig, 3)
fig, ax = figbase(3.7)
sel = [i for i in [0,1,6,7,5,4] if i<len(modelos)] if len(modelos)==9 else [0,1,6,5,4]
labels = (['AR(2), principal','ARMA(1,1)','AR(2), fuente alternativa','AR(2), desde 2021','AR(2), incluido Paita','AR(2), solamente Callao'] if len(modelos)==9 else ['AR(2), principal','ARMA(1,1)','AR(2), desde 2021','AR(2), incluido Paita','AR(2), solamente Callao'])
mm = modelos.iloc[sel]
x = mm.coef_post.to_numpy()
lo = mm.ic95_inferior.to_numpy()
hi = mm.ic95_superior.to_numpy()
yy = np.arange(len(x))
ax.errorbar(x, yy, xerr=np.vstack([x - lo, hi - x]), fmt='o', capsize=3, markersize=5)
ax.set_yticks(yy, labels)
ax.invert_yaxis()
ax.axvline(0, linestyle='--', linewidth=1)
ax.set_xlabel('Cambio de nivel estimado (puntos HHI), IC del 95 %')
ax.grid(axis='x', alpha=0.15)
save(fig, 4)
fig, ax = figbase(3.2)
idx = np.arange(5)
w = 0.35
ax.bar(idx - w / 2, q.HHI_normalizado_2024, w, label='2024')
ax.bar(idx + w / 2, q.HHI_normalizado_2025, w, label='2025')
ax.set_xticks(idx, [meses[i - 1] for i in CONFIG['comparacion_meses']])
ax.set_ylabel('HHI normalizado por participantes')
ax.set_xlabel('Mes de comparación')
ax.set_ylim(0, 0.23)
ax.legend(frameon=False)
save(fig, 5)
assert len(list(FIGURAS.glob('figura_??.png'))) == 5
assert len(list(FIGURAS.glob('figura_??.pdf'))) == 5
print('Guardadas cinco figuras en PDF vectorial y PNG.')
for i in range(1, 6):
    print('Figura', i, ':', FIGURAS / f'figura_{i:02d}.png')

## 17. Tablas exportables y biblioteca de fuentes

Se conservan los resultados en CSV y Excel, acompañados de los identificadores y enlaces de las publicaciones originales. Cada PDF o Excel de la APN dispone de una versión tabular del segmento utilizado, preparada para su consulta en una futura biblioteca de datos. Esta asociación mantiene la procedencia de los documentos, pero no modifica los datos con fines de presentación.

In [ ]:
with pd.ExcelWriter(RESULTADOS/'RESULTADOS_ECONOMETRICOS.xlsx',engine='openpyxl') as escritor:
    modelos.to_excel(escritor,sheet_name='modelos',index=False)
    coeficientes.to_excel(escritor,sheet_name='coeficientes_AR2',index=False)
    q.to_excel(escritor,sheet_name='descomposicion',index=False)
    BCRP.to_excel(escritor,sheet_name='BCRP',index=False)
    concordancia.to_excel(escritor,sheet_name='concordancia_PDF_Excel',index=False)
q.to_csv(RESULTADOS/'descomposicion_junio_octubre.csv',index=False,encoding='utf-8-sig')
modelos.to_csv(RESULTADOS/'modelos_y_diagnosticos.csv',index=False,encoding='utf-8-sig')
coeficientes.to_csv(RESULTADOS/'coeficientes_AR2.csv',index=False,encoding='utf-8-sig')

def crear_libro(destino,datos,metadata):
    wb=Workbook(); meta=wb.active;meta.title='Procedencia';meta.append(['Campo','Información'])
    for nombre,valor in metadata.items():meta.append([nombre,str(valor)])
    ws=wb.create_sheet('Datos procesados')
    if datos.empty:ws.append(['Sin tabla tabular extraíble; pendiente de extracción verificada'])
    else:
        ws.append(list(datos.columns))
        for fila in datos.itertuples(index=False,name=None):
            ws.append([v.item() if isinstance(v,np.generic) else (None if pd.isna(v) else v) for v in fila])
    for hoja in wb:
        hoja.sheet_view.showGridLines=False;hoja.freeze_panes='A2'
        for celda in hoja[1]:
            celda.font=Font(name='Aptos',bold=True,color='FFFFFF')
            celda.fill=PatternFill('solid',fgColor='203A56')
        for columna in hoja.columns:
            largo=max(len(str(c.value or '')) for c in columna)
            hoja.column_dimensions[get_column_letter(columna[0].column)].width=min(60,max(13,largo+2))
    wb.save(destino)

biblioteca = []
for r in DESCARGAS.itertuples(index=False):
    original=ORIG_APN/r.archivo
    trozo=APN.loc[APN.archivo_original==r.archivo].copy()
    estado_extraccion = 'pendiente de extracción tabular verificada' if trozo.empty else 'datos extraídos'
    met={'Institución':'Autoridad Portuaria Nacional','Periodo':r.periodo,'Documento':r.archivo,
         'Estado de extracción':estado_extraccion,
         'URL oficial':r.url,'SHA256':r.sha256_descargado,'Registros de tráfico':len(trozo)}
    destino=WEB_EXCEL/(original.stem+'_datos.xlsx')
    crear_libro(destino,trozo,met)
    biblioteca.append({'institucion':'APN','periodo':r.periodo,'fuente_url':r.url,
                       'archivo_original':str(original.relative_to(CARPETA_RAIZ)),
                       'excel_procesado':str(destino.relative_to(CARPETA_RAIZ)),
                       'sha256':r.sha256_descargado,'registros':len(trozo),
                       'estado_extraccion':estado_extraccion})
for codigo,columna in CONFIG['codigos_bcrp'].items():
    original=ORIG_BCRP/(codigo+'.json')
    url=f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{codigo}/json/{CONFIG["inicio"]}/{CONFIG["fin"]}/esp'
    destino=WEB_EXCEL/(codigo+'_datos.xlsx')
    crear_libro(destino,BCRP[['mes',columna]],{'Institución':'BCRP','Serie':codigo,'URL oficial':url,
                                            'SHA256':hashlib.sha256(original.read_bytes()).hexdigest()})
    biblioteca.append({'institucion':'BCRP','periodo':f'{CONFIG["inicio"]} a {CONFIG["fin"]}',
                       'fuente_url':url,'archivo_original':str(original.relative_to(CARPETA_RAIZ)),
                       'excel_procesado':str(destino.relative_to(CARPETA_RAIZ)),
                       'sha256':hashlib.sha256(original.read_bytes()).hexdigest(),'registros':len(BCRP)})
pd.DataFrame(biblioteca).to_csv(WEB/'catalogo_fuentes.csv',index=False,encoding='utf-8-sig')
(WEB/'catalogo_fuentes.json').write_text(json.dumps(biblioteca,ensure_ascii=False,indent=2),encoding='utf-8')
print('Archivo de resultados Excel y',len(biblioteca),'libros de fuente generados.')

## 18. Verificación de integridad y archivos de salida

Se registra la huella SHA-256 de cada archivo descargado y generado y se comprueba que el calendario, el HHI principal, las especificaciones y las figuras estén completos. La comprobación de versiones históricas permite identificar posibles revisiones de las fuentes. Una discrepancia de huella exige revisar el documento antes de asumir que los resultados son directamente comparables con los del artículo.

In [ ]:
assert len(DESCARGAS)==56
assert len(list(ORIG_BCRP.glob('*.json')))==4
assert len(d)==139 and d.hhi_sys.notna().sum()==132
assert len(modelos)==8 and modelos.convergio.all()
assert int(base.apta_para_modelo.sum())==132
assert d.apta_para_modelo.eq(d.hhi_sys.notna()).all()
assert len(list(FIGURAS.glob('figura_??.png')))==5
assert len(list(WEB_EXCEL.glob('*.xlsx')))==60
pd.DataFrame(LOG).to_csv(AUDITORIA/'registro_consultas_http.csv',index=False,encoding='utf-8-sig')
registros=[]
for archivo in sorted(CARPETA_RAIZ.rglob('*')):
    if archivo.is_file() and archivo.name!='manifest_sha256.csv':
        registros.append({'archivo':str(archivo.relative_to(CARPETA_RAIZ)),
                          'sha256':hashlib.sha256(archivo.read_bytes()).hexdigest(),
                          'bytes':archivo.stat().st_size})
pd.DataFrame(registros).to_csv(AUDITORIA/'manifest_sha256.csv',index=False,encoding='utf-8-sig')
print('Procesamiento y estimación finalizados.')
print('Fuentes APN:',len(DESCARGAS),'| Series BCRP:',len(CONFIG['codigos_bcrp']))
print('HHI válido:',d.hhi_sys.notna().sum(),'| Modelos:',len(modelos),'| Figuras:',len(list(FIGURAS.glob('*.png'))))
print('Carpeta:',CARPETA_RAIZ)

## Archivos de investigación

La carpeta de trabajo conserva las publicaciones originales de la APN y las respuestas de la API del BCRP, los enlaces encontrados con el rastreo institucional, los registros HTTP, la base integrada, la descomposición, las estimaciones, las figuras y los Excel de consulta por fuente. El catálogo de procedencia distingue los documentos utilizados de sus transformaciones; los resultados pueden someterse a una nueva verificación descargando las publicaciones oficiales con este mismo cuaderno.